In [ ]:
import torch
if torch.cuda.is_available():
    print(f" SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
else:
    print(" FAILED! PyTorch is still stuck on CPU.")

 SUCCESS! PyTorch found the GPU: Tesla T4


In [ ]:
!pip install -q transformers accelerate onnx onnxscript torchvision

import torch
from torchvision.datasets import GTSRB
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
import numpy as np
from google.colab import files
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Download GTSRB via PyTorch (Built-in)
print("Downloading GTSRB Dataset...")
train_data_raw = GTSRB(root='./gtsrb_data', split='train', download=True)

# GTSRB has 43 classes (0 to 42)
num_classes = 43
id2label = {i: f"GTSRB_Class_{i}" for i in range(num_classes)}
label2id = {v: k for k, v in id2label.items()}

# 2. Load Pre-trained ViT
# We use a standard 224x224 ViT base model for classification
model_id = "google/vit-base-patch16-224-in21k"
processor = ViTImageProcessor.from_pretrained(model_id)
model = ViTForImageClassification.from_pretrained(
    model_id,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

# 3. Create a Hugging Face compatible Dataset
class GTSRBDataset(torch.utils.data.Dataset):
    def __init__(self, pt_dataset, processor):
        self.pt_dataset = pt_dataset
        self.processor = processor

    def __len__(self):
        return len(self.pt_dataset)

    def __getitem__(self, idx):
        image, label = self.pt_dataset[idx]
        # Processor handles resizing to 224x224 and normalization automatically!
        pixel_values = self.processor(image.convert("RGB"), return_tensors="pt")["pixel_values"].squeeze(0)
        return {"pixel_values": pixel_values, "labels": torch.tensor(label)}

print("Formatting data for Vision Transformer...")
# For speed in this demo, we use a subset. Remove the Subset logic to train on all 26,000 images!
# subset_indices = torch.randperm(len(train_data_raw))[:5000]
# train_dataset = GTSRBDataset(torch.utils.data.Subset(train_data_raw, subset_indices), processor)
train_dataset = GTSRBDataset(train_data_raw, processor)

# 4. Train the Classifier
training_args = TrainingArguments(
    output_dir="./vit_gtsrb",
    per_device_train_batch_size=32,
    num_train_epochs=3, # 3 Epochs is plenty for classification transfer learning
    fp16=True,
    logging_steps=50,
    learning_rate=2e-5,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("\n--- Starting GTSRB Training ---")
trainer.train()

# 5. Export to ONNX
print("\nExporting GTSRB Classifier to ONNX...")
dummy_input = torch.randn(1, 3, 224, 224, device=device)
onnx_filename = "vit_gtsrb_classifier.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    input_names=["pixel_values"],
    output_names=["logits"],
    opset_version=18
)

print(f"Successfully exported {onnx_filename}!")
files.download(onnx_filename)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 11.5 MB/s eta 0:00:00
Using device: cuda


100%|██████████| 187M/187M [00:17<00:00, 11.0MB/s]


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

...

[torch.onnx] Obtain model graph for `ViTForImageClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ViTForImageClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Successfully exported vit_gtsrb_classifier.onnx!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>